<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step01PrepareClip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 01 · Prepare the clip

Probes one calving-pen video and copies the window with the calving into the group's Shared Drive, `CalvingPenPipelineOutputs/<video>/`:

- `ReferenceClip.mp4`: the window you set below (a stream copy, a few minutes)
- `DevClip.mp4`: optional short re-encoded piece (set `DevStart`)
- `Probe.json`: video + clip info and the settings used

**Already done for the reference video**; only needed for a new video. Cell 2 lists the videos on the Drive: copy the `VideoFile` name of yours. Not sure when the calving is? Run cell 3 once with any `ClipStart`: it first saves `Timeline.jpg` (one picture every 15 min), then set `ClipStart` / `ClipEnd` and rerun. Rerunning **overwrites** these files. No GPU needed.
The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells. Save your run with File > Save a copy in Drive, never back to GitHub.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import PrepareClips, SaveTimelineSheet
from CalvingPipeline.Paths import FindVideo, ListVideos, OutputRoot
from IPython.display import display

In [ ]:
# 2. Settings: the only cell to edit.
display(ListVideos())    # the calving videos on the Shared Drive: pick one, copy its VideoFile name below

VideoFile = "2026-09-26-CalvingPen3(1).avi"   # file name only; found in "CalvingPenVideos " (or anywhere in the group folder)
VideoStem = None         # output folder name; None = the file name without .avi. The reference video keeps
                         # its old name: "2026-09-26 00-00-00 Calving Pen 2"

ClipStart = "00:00:00"   # HH:MM:SS into the video where the cow (calving) starts; see the timeline sheet below
ClipEnd = None           # HH:MM:SS where it ends; None = to the end of the video
TimelineEveryMinutes = 15  # cell 3 also saves Timeline.jpg: one picture every 15 min, to choose ClipStart/End
DevStart = None          # leave None (an optional short re-encoded test piece; slow)
DevSeconds = None

VideoPath = FindVideo(VideoFile)
VideoOutputDir = OutputRoot / (VideoStem or VideoPath.stem)
print("Video:", VideoPath, "\nOutputs:", VideoOutputDir)

In [ ]:
# 3. Run: save the timeline sheet, then probe + cut the clip (overwrites earlier output).
from IPython.display import Image
SaveTimelineSheet(VideoPath, VideoOutputDir / "Timeline.jpg", EveryMinutes=TimelineEveryMinutes)
display(Image(str(VideoOutputDir / "Timeline.jpg"), width=1200))
# Not sure about ClipStart / ClipEnd yet? Look at Timeline.jpg, set them in cell 2, rerun cells 2-3.
Summary = PrepareClips(VideoPath, VideoOutputDir, ClipStart, ClipEnd, DevStart, DevSeconds)
Summary

## About this step

**Tool:** ffmpeg (no model). The clip is a *stream copy*: the original pictures are copied, not re-encoded, so an hour takes minutes.

**What we found on the reference video** (`2026-09-26 00-00-00 Calving Pen 2.avi`, window 03:35:11 to the end):
- The AVI header claims **150 fps**; the real rate is **30 fps** (the recorder writes ~5 index entries per frame). The copied clip reports the correct 30 fps.
- `ReferenceClip.mp4`: 1:02:01, 1920×1080, 857 MB. About 93 % of consecutive frames are identical: the camera updates the picture only ~2 times a second.
- Re-encoding (the old `DevClip`) is slow because of the fake 150 fps; leave `DevStart = None` unless a short re-encoded piece is really needed.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.